In [19]:
## layer 1, understand strides and kernel_size concepts
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import os
# https://flypix.ai/blog/image-recognition-algorithms/ why I choosed only first few layers for feature extraction, upto middle layers are sufficient to provide information about eyes
# In higher layers of the network, detailed pixel information is lost whilethe high level content of the image is preserved. Clear Explanation is here(https://ai.stackexchange.com/questions/30038/why-do-we-lose-detail-of-an-image-as-we-go-deeper-into-a-convnet)
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super(ResNetFeatureExtractor, self).__init__()
        resnet = models.resnet18(pretrained=True)
        # print(resnet18)

        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-3])
        
        
        self.feature_extractor[-1][0].conv1.stride = (1, 1) # normally resnet has stride = 2 in layer 3 this will downsample from 8x8 to 4x4. This is to avoid this. More spatil features are preserved
        self.feature_extractor[-1][0].downsample[0].stride = (1, 1)

        self.reduce_channels = nn.Conv2d(256, 128, kernel_size=1) # kernel_size 1 only changes the number of channels and doesn't mess with the spatial size

    def forward(self, x):
        x = self.feature_extractor(x)  
        x = self.reduce_channels(x)   
        return x

feature_extractor = ResNetFeatureExtractor()
feature_extractor.eval()  

transform = transforms.Compose([ # pre processing the images to match the resnet's training statistics
    transforms.Resize((128, 128)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalization of the pixel values
])

def extract_features(image_path):
    img = Image.open(image_path).convert("RGB") 
    img = transform(img).unsqueeze(0) # apply the transformations
    # with torch.no_grad():
    features = feature_extractor(img)  
    
    return features

dataset_path = "./Test Dataset/Output/"

left_eye_img = os.path.join(dataset_path, "webcam_r/left_eye/left_eye_0000.jpg")
right_eye_img = os.path.join(dataset_path, "webcam_r/right_eye/right_eye_0000.jpg")
face_img = os.path.join(dataset_path, "webcam_r/face/face_0000.jpg")


left_eye_features = extract_features(left_eye_img)  
right_eye_features = extract_features(right_eye_img)  
face_features = extract_features(face_img) 


print(left_eye_features.shape)
print(right_eye_features.shape)
print(face_features.shape)

torch.Size([1, 128, 16, 16])
torch.Size([1, 128, 16, 16])
torch.Size([1, 128, 16, 16])


In [3]:
import torch 


# input of layer 2
Leye = torch.rand(128, 8, 8)
print(Leye)

Reye = torch.rand(128, 8, 8)
print(Reye)

FaceData = torch.rand(32, 8, 8)
print(FaceData)

class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        

        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        self.gn = torch.nn.GroupNorm(3, 6)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?

        # layer 3:.......................
    
    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((torch.tensor(left_eye), torch.tensor(right_eye), torch.tensor(face)), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        return out

model = TinyModel()
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)


tensor([[[0.5710, 0.9573, 0.8388,  ..., 0.4309, 0.6644, 0.3738],
         [0.4550, 0.9207, 0.3412,  ..., 0.0932, 0.2880, 0.1926],
         [0.7916, 0.8939, 0.8359,  ..., 0.6462, 0.1381, 0.6904],
         ...,
         [0.0104, 0.5017, 0.1842,  ..., 0.5362, 0.9441, 0.2077],
         [0.5548, 0.9118, 0.8449,  ..., 0.0899, 0.1121, 0.2897],
         [0.7365, 0.4915, 0.4410,  ..., 0.0795, 0.9189, 0.3403]],

        [[0.5119, 0.1292, 0.6125,  ..., 0.5156, 0.3652, 0.9493],
         [0.9338, 0.7321, 0.2504,  ..., 0.1962, 0.4210, 0.7808],
         [0.9430, 0.5609, 0.2115,  ..., 0.3353, 0.9994, 0.5822],
         ...,
         [0.2121, 0.2898, 0.7662,  ..., 0.5397, 0.2893, 0.7222],
         [0.6827, 0.8154, 0.9964,  ..., 0.4438, 0.2510, 0.8919],
         [0.1765, 0.3628, 0.9784,  ..., 0.1422, 0.8220, 0.3777]],

        [[0.6562, 0.0836, 0.2420,  ..., 0.9649, 0.9346, 0.7702],
         [0.3285, 0.0924, 0.1178,  ..., 0.1298, 0.7029, 0.4461],
         [0.8435, 0.2155, 0.7781,  ..., 0.3284, 0.9975, 0.

C:\Users\lingn\AppData\Local\Temp\ipykernel_11480\3743870404.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  concate = torch.cat((torch.tensor(left_eye), torch.tensor(right_eye), torch.tensor(face)), 1)  # dim = 0 or 1?  only channel dim changes?


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 128 but got size 32 for tensor number 2 in the list.